# COMM061 : NLP
# Coursework: BESSTIE Sequence Classification 
# Group 29
# **Task:** Sentiment and Sarcasm detection across three English varieties (en-AU, en-IN, en-UK)  
# **Dataset:** surrey-nlp/BESSTIE-CW-26 (6,243 samples)

In [ ]:
import sys
import os
import warnings

# Add the current directory to the path to allow importing from src
sys.path.insert(0, os.path.abspath('.'))

# Filter out warnings
warnings.filterwarnings('ignore')

# Import constants from your config file
from src.config import SEEDS, VARIETIES, TASKS, RESULTS_DIR, FIGURES_DIR

# Print configuration details
print(f"Seeds: {SEEDS}")
print(f"Varieties: {VARIETIES}")
print(f"Tasks: {TASKS}")
print(f"Results dir: {RESULTS_DIR}")
print(f"Figures dir: {FIGURES_DIR}")

---## Q1: Data Analysis and Visualisation (15 marks)*Owner: M1 · Max 4 pages in report*

### Q1.1 — Label DistributionsVisualise Sentiment and Sarcasm class distributions across all three varieties.  Comment on the sarcasm class imbalance (en-IN ~7%, en-UK ~8% vs en-AU ~29%).

In [ ]:
from src.data.loader import get_split_stats
from src.data.analysis import plot_label_distributions, plot_sarcasm_imbalance_per_split

# Get and print split statistics
stats = get_split_stats()
print(stats.to_string(index=False))

# Generate and save distribution plots
label_stats = plot_label_distributions(save=True)
plot_sarcasm_imbalance_per_split(save=True)

### Q1.2 — Vocabulary OverlapCalculate overlap between en-UK (inner-circle) and en-IN (outer-circle) using:- Jaccard similarity on unique token sets- Cosine similarity on mean TF-IDF vectors

In [ ]:
from src.data.analysis import compute_vocab_overlap

# Compute the vocabulary overlap between the two language varieties
overlap = compute_vocab_overlap(variety_a="en-UK", variety_b="en-IN")

# Optional: You might want to print the result to see the output
print(overlap)

### Bonus — POS Tag AnalysisCompare POS tag distributions between sarcastic and non-sarcastic samples.

In [ ]:
from src.data.analysis import pos_distribution_by_label

# Analyze POS distribution for UK variety
pos_distribution_by_label(task="Sarcasm", variety="en-UK", save=True)

# Analyze POS distribution for AU variety
pos_distribution_by_label(task="Sarcasm", variety="en-AU", save=True)

---## Q2: Experimentation (40 marks)*Max 6 pages in report for Q2.1 + Q2.2 + Q2.3 combined*

### Q2.1 — Baseline vs Transformer (10 marks)*Owner: M2*Compare classical baselines (TF-IDF + LogReg, TF-IDF + SVM) against fine-tuned RoBERTa-base.  Both tasks, 2 seeds each. Focus the report on *why* pre-training helps (or doesn't).

#### Classical Baselines

In [ ]:
from src.models.classical import run_classical_baselines

# Run the classical baseline models and store the results in a DataFrame
df_classical = run_classical_baselines()

# Display the DataFrame
df_classical

#### Fine-tuned RoBERTa

In [ ]:
from src.models.transformer import run_transformer_experiment

# Run the RoBERTa (or similar transformer) experiment
df_roberta = run_transformer_experiment()

# Display the resulting DataFrame
df_roberta

#### Q2.1 — Comparison

In [ ]:
import pandas as pd

# Combine classical + transformer results
q21_classical = pd.read_csv(RESULTS_DIR / "q21_classical_baselines.csv")
q21_roberta = pd.read_csv(RESULTS_DIR / "q21_roberta.csv")
q21_all = pd.concat([q21_classical, q21_roberta], ignore_index=True)

# Average across seeds for the summary table
summary = q21_all.groupby(["model", "task", "variety"])[
    ["macro_f1", "precision_macro", "recall_macro"]
].mean().round(4)

print(summary.to_string())

In [ ]:
from src.evaluation.visualisation import plot_model_comparison

# Generate and save comparison plot for Sentiment task
plot_model_comparison(q21_all, task="Sentiment", save_name="q21_comparison_sentiment.png")

# Generate and save comparison plot for Sarcasm task
plot_model_comparison(q21_all, task="Sarcasm", save_name="q21_comparison_sarcasm.png")

### Q2.2 — Cross-Variety Evaluation (15 marks)*Owner: M2*Train RoBERTa per variety for Sentiment. Test each on all three test sets.  Analyse: does AU→UK transfer better than AU→IN?

In [ ]:
from src.models.transformer import run_cross_variety_experiment

# Run the cross-variety experiment for the Sentiment task
# This returns both the performance matrix and a detailed DataFrame
matrix, df_cross = run_cross_variety_experiment(task="Sentiment")

# Display the averaged Macro-F1 performance matrix
print("\nCross-variety matrix (Macro-F1, averaged across seeds):")
print(matrix.round(4))

In [ ]:
from src.evaluation.visualisation import plot_cross_variety_heatmap

# Generate and save the heatmap for the Cross-variety Sentiment analysis
plot_cross_variety_heatmap(matrix, task="Sentiment", save_name="q22_heatmap.png")

### Q2.3 — LoRA Adapters (15 marks)*Owner: M3*LoRA adapters on Qwen2.5-1.5B-Instruct per variety for Sarcasm detection.  Compare UK/AU/IN adapters on all test sets.

In [ ]:
from src.models.lora_adapter import run_lora_experiment

# Set max_samples=500 to reduce GPU time if needed (spec allows this)
# Running with max_samples=None uses the full dataset
df_lora, adapter_paths = run_lora_experiment(max_samples=None)

# Display the results DataFrame
df_lora

In [ ]:
# Build the LoRA cross-variety matrix
lora_results = pd.read_csv(RESULTS_DIR / "q23_lora.csv")

# Group and unstack to create the matrix format
lora_matrix = lora_results.groupby(["train_variety", "variety"])["macro_f1"].mean().unstack()

print("LoRA adapter cross-variety matrix (Macro-F1):")
print(lora_matrix.round(4))

---## Q3: Evaluation (15 marks)*Owner: All · Max 5 pages in report*Report Macro-F1 (primary), precision, recall per class for all experiments.  Confusion matrices for best models. Check for majority-class bias.

In [ ]:
from src.evaluation.visualisation import compile_all_results

# Compile all results from classical, transformer, and LoRA experiments
all_results = compile_all_results()

# Print summary statistics for the compiled data
print(f"Total experiment rows: {len(all_results)}")
print(f"Experiments: {all_results['experiment'].unique()}")
print(f"Models: {all_results['model'].unique()}")

# Display the first 20 rows of the consolidated DataFrame
all_results.head(20)

### Summary Table — All Experiments (averaged across seeds)

In [ ]:
# Create the final summary table with means and standard deviations
summary_table = all_results.groupby(
    ["experiment", "model", "task", "variety"]
)[["macro_f1", "precision_macro", "recall_macro", "f1_class_0", "f1_class_1"]].agg(
    ["mean", "std"]
).round(4)

# Print the complete summary table
print(summary_table.to_string())

### Majority Class Bias Check

In [ ]:
# Check if any model only predicted the majority class
biased = all_results[all_results["majority_class_only"] == True]

if len(biased) > 0:
    print("WARNING: The following runs only predicted one class:")
    print(biased[["experiment", "model", "task", "variety", "seed"]].to_string(index=False))
else:
    print("No majority-class-only predictions detected across all experiments.")

### Confusion Matrices — Best Models

In [ ]:
# NOTE: Confusion matrices need y_true/y_pred, not aggregated CSVs.
# Re-run the best model's predictions or save them during training.

# Example for the best RoBERTa model:
~                   save_name="q3_cm_roberta_sarcasm_uk.png")

print("Generate confusion matrices by re-running best model predictions.")
print("See src/evaluation/visualisation.py: plot_confusion_matrix()")

---## Q4: Error Analysis and Few-Shot Prompting (10 marks)*Owner: M3 · Max 4 pages in report*1. Extract 10 misclassified sarcasm examples from best LoRA adapter2. Write explanations for 4 of them (human reasoning — not generated)3. Build a few-shot prompt with those 4 explained examples4. Test remaining 6 with the prompt5. Compare before/after

In [ ]:
from src.models.prompting import run_error_analysis

# Replace with the actual best adapter path from Q2.3
# Picking the first one as a placeholder; in practice, select the one with the highest F1
best_adapter_path = list(adapter_paths.values())[0]  
best_variety = "en-UK"  # Ensure this matches the variety the adapter was trained for

# Run the error analysis using a prompting-based approach (likely LLM-driven)
q4_results = run_error_analysis(best_adapter_path, best_variety)

In [ ]:

# Extract the errors from the results dictionary
errors_df = pd.DataFrame(q4_results["errors"])

# Print the top 10 misclassified examples for qualitative analysis
print("Misclassified examples:")
print(errors_df[["text", "true_name", "pred_name"]].head(10).to_string(index=False))

In [ ]:
# Show before/after comparison for the 6 test examples
print("\nFew-shot prompt results (remaining 6 examples):")
print(q4_results["fewshot_results"][["text", "true_name", "original_pred", "fewshot_pred", "improved"]].to_string(index=False))

# Calculate improvement statistics
improved = q4_results["fewshot_results"]["improved"].sum()
total = len(q4_results["fewshot_results"])
print(f"\nImproved: {improved}/{total} predictions corrected by few-shot prompting")

---## Q5: Deployment (20 marks)

### Q5.1 — Deployment Endpoint (15 marks)*Owner: M4 · Max 5 pages in report*Gradio app with text input + variety dropdown.  Backend loads the appropriate LoRA adapter via from_pretrained().

In [ ]:
from src.deployment.app import build_app

# Uncomment to launch the app locally:
# app = build_app()
# app.launch(share=True)

print("Deployment app defined in src/deployment/app.py")
print("Launch with: python -m src.deployment.app")
print("Or uncomment the lines above to launch from this notebook.")

### Q5.2 — Efficiency Analysis (5 marks)*Owner: M4 · Max 1 page in report*Compare inference time: SVM (CPU) vs RoBERTa (GPU) vs LoRA-LLM (GPU).

In [ ]:
from src.deployment.benchmark import run_benchmarks

# You need to pass actual predict functions from each model.
# Example (fill in after models are trained):
def svm_predict(text):
     pipe = joblib.load("path/to/svm_pipeline.pkl")
     return pipe.predict([text])[0]

def roberta_predict(text):
     return predict_with_model(roberta_model, roberta_tokenizer, [text])[0]

def lora_predict(text):
     from src.deployment.app import predict
     return predict(text, "en-UK")

timing_df = run_benchmarks(svm_predict, roberta_predict, lora_predict)
timing_df

print("Fill in predict functions after all models are trained.")
print("See src/deployment/benchmark.py for the benchmarking framework.")

---## References- Srirag et al. (2025). "BESSTIE: A Benchmark for Sentiment and Sarcasm Classification for Varieties of English." Findings of ACL 2025.- Abercrombie & Hovy (2016). "Putting Sarcasm Detection into Context." ACL Student Research Workshop.- Singh et al. (2025). "Nek Minit: Harnessing Pragmatic Metacognitive Prompting for Explainable Sarcasm Detection." arXiv:2505.15095.- Ribeiro et al. (2016). "Why Should I Trust You?" KDD 2016. (LIME)- Lundberg & Lee (2017). "A Unified Approach to Interpreting Model Predictions." NeurIPS 2017. (SHAP)